<a href="https://colab.research.google.com/github/samary12/bayan-project/blob/main/notebooks/05_arabic_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# اليوم الثالث — مختبر 4: معالجة العربية | Day 3 — Lab 4: Arabic NLP

**المدربة / Instructor:** ميعاد المري — Meaad Al-Marri  
**المسار:** 🟢 Core → 🔵 Explore → 🟣 Distinction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/almiyead-rgb/bayan-applied-nlp-course/blob/main/notebooks/05_arabic_nlp.ipynb)

**الهدف:** بناء عقد معالجة عربي يحفظ النص الأصلي للعرض، وينشئ نسخة مستقلة للنموذج باستخدام CAMeL Tools وprofile مسماة ومثبتة الإصدار.

**Goal:** preserve an authoritative display copy and derive a versioned model/search copy with an explicit Arabic normalisation profile.


## قبل التشغيل | Before you run

- استخدم بيانات الدورة فقط؛ لا تلصق بيانات مستفيدين حقيقية.
- نفّذ `Runtime → Run all`، ولا تتجاوز خلية فاشلة.
- لا نعتبر `variant` في البيانات تنبؤ لهجة؛ هو وسم تعليمي أنشئ مع العينة.
- Arabizi في هذا الدفتر **heuristic flag** شفاف، وليس مصنفًا مدربًا.
- Core يعمل على CPU ولا يحتاج GPU أو اشتراكًا.

**Evidence labels:** all counts in this notebook are `COURSE_FIXTURE`; they describe the supplied synthetic sample, not population statistics.


In [2]:
# خلية التجهيز — تثبت الحزمة المطلوبة فقط إذا اختلف الإصدار.
import importlib.metadata
import subprocess
import sys

REQUIRED = {
    "camel-tools": "1.6.0",
    "transformers": "5.15.1",
    "tokenizers": "0.22.2",
    "scikit-learn": "1.9.0",
}
to_install = []
for distribution, expected in REQUIRED.items():
    try:
        current = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        current = None
    if current != expected:
        to_install.append(f"{distribution}=={expected}")

if to_install:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *to_install]
    )

assert importlib.metadata.version("camel-tools") == "1.6.0"
print("SETUP=PASS", {name: importlib.metadata.version(name) for name in REQUIRED})


SETUP=PASS {'camel-tools': '1.6.0', 'transformers': '5.15.1', 'tokenizers': '0.22.2', 'scikit-learn': '1.9.0'}


In [3]:
import csv
import io
import json
import re
import unicodedata
import urllib.request
from collections import Counter
from pathlib import Path

from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import (
    normalize_alef_ar,
    normalize_alef_maksura_ar,
    normalize_unicode,
)

PROFILE_VERSION = "1.0.0"
DATA_KIND = "COURSE_FIXTURE"
print("IMPORTS=PASS")


IMPORTS=PASS


## 1) البيانات وعقدها | Data contract

يحاول الدفتر قراءة CSV العام من GitHub. إذا انقطع الوصول إلى الملف، يستخدم نسخة مطابقة مدمجة حتى يستمر المختبر. كل صف له `variant` تعليمي: `MSA` أو `Gulf` أو `Arabizi`.

The fallback protects the lesson from a raw-file outage; it does not replace the CAMeL Tools dependency.


In [4]:
DATA_URL = "https://raw.githubusercontent.com/almiyead-rgb/bayan-applied-nlp-course/main/data/sample/bayan_day3_arabic.csv"
FALLBACK_ROWS = [{'record_id': 'A-001', 'language': 'ar', 'variant': 'Gulf', 'channel': 'chat', 'topic': 'digital_service', 'text': 'وش سبب تعليق البوابة كل ما أرفع الملف؟'}, {'record_id': 'A-002', 'language': 'ar', 'variant': 'Gulf', 'channel': 'chat', 'topic': 'digital_service', 'text': 'ما وصلني رمز التحقق للحين'}, {'record_id': 'A-003', 'language': 'ar', 'variant': 'Gulf', 'channel': 'chat', 'topic': 'transport', 'text': 'الباص تأخر علينا والموعد راح'}, {'record_id': 'A-004', 'language': 'ar', 'variant': 'Gulf', 'channel': 'chat', 'topic': 'transport', 'text': 'وين ألقى مسار الحافلة الجديد؟'}, {'record_id': 'A-005', 'language': 'ar', 'variant': 'Gulf', 'channel': 'chat', 'topic': 'health', 'text': 'أبي أغير موعد العيادة لبكرة'}, {'record_id': 'A-006', 'language': 'ar', 'variant': 'Gulf', 'channel': 'chat', 'topic': 'permit', 'text': 'طلبي واقف من أسبوع وش المطلوب؟'}, {'record_id': 'A-007', 'language': 'ar', 'variant': 'Gulf', 'channel': 'web', 'topic': 'permit', 'text': 'انرفض المرفق مع إنه واضح'}, {'record_id': 'A-008', 'language': 'ar', 'variant': 'Gulf', 'channel': 'chat', 'topic': 'health', 'text': 'النتيجة للحين ما نزلت في التطبيق'}, {'record_id': 'A-009', 'language': 'ar', 'variant': 'MSA', 'channel': 'web', 'topic': 'digital_service', 'text': 'تعذر تسجيل الدخول إلى البوابة الإلكترونية'}, {'record_id': 'A-010', 'language': 'ar', 'variant': 'MSA', 'channel': 'web', 'topic': 'digital_service', 'text': 'يظهر خطأ عند رفع المستند المطلوب'}, {'record_id': 'A-011', 'language': 'ar', 'variant': 'MSA', 'channel': 'web', 'topic': 'transport', 'text': 'تأخرت الحافلة عن الموعد المحدد'}, {'record_id': 'A-012', 'language': 'ar', 'variant': 'MSA', 'channel': 'web', 'topic': 'transport', 'text': 'أرغب في معرفة أقرب محطة للحافلات'}, {'record_id': 'A-013', 'language': 'ar', 'variant': 'MSA', 'channel': 'web', 'topic': 'health', 'text': 'أرجو إعادة جدولة موعد العيادة'}, {'record_id': 'A-014', 'language': 'ar', 'variant': 'MSA', 'channel': 'web', 'topic': 'health', 'text': 'لم تظهر نتيجة الفحص في الملف الصحي'}, {'record_id': 'A-015', 'language': 'ar', 'variant': 'MSA', 'channel': 'web', 'topic': 'permit', 'text': 'ما زالت حالة التصريح قيد المراجعة'}, {'record_id': 'A-016', 'language': 'ar', 'variant': 'MSA', 'channel': 'web', 'topic': 'permit', 'text': 'تم رفض الوثيقة المرفقة دون توضيح'}, {'record_id': 'A-017', 'language': 'ar', 'variant': 'MSA', 'channel': 'web', 'topic': 'digital_service', 'text': 'إِدَارَةُ الحِساب لا تعمل بعد التحديث'}, {'record_id': 'A-018', 'language': 'ar', 'variant': 'Gulf', 'channel': 'chat', 'topic': 'transport', 'text': 'الخـدمة متأخرة مرررة اليوم'}, {'record_id': 'A-019', 'language': 'ar', 'variant': 'Arabizi', 'channel': 'chat', 'topic': 'digital_service', 'text': 'ma wasalni code 3al jawal'}, {'record_id': 'A-020', 'language': 'ar', 'variant': 'Arabizi', 'channel': 'chat', 'topic': 'transport', 'text': 'al bus ta5ar 2 hours'}]

try:
    with urllib.request.urlopen(DATA_URL, timeout=15) as response:
        rows = list(csv.DictReader(io.StringIO(response.read().decode("utf-8"))))
    data_source = "github"
except Exception as exc:
    rows = FALLBACK_ROWS
    data_source = f"embedded_fallback:{type(exc).__name__}"

required_columns = {"record_id", "language", "variant", "channel", "topic", "text"}
assert len(rows) == 20
assert required_columns <= set(rows[0])
assert len({row["record_id"] for row in rows}) == len(rows)
assert {row["variant"] for row in rows} == {"MSA", "Gulf", "Arabizi"}
print({"data_kind": DATA_KIND, "source": data_source, "rows": len(rows)})


{'data_kind': 'COURSE_FIXTURE', 'source': 'github', 'rows': 20}


## 2) نسختان للنص، لا نسخة واحدة | Two-copy contract

| النسخة | الغرض | هل نكتب فوقها؟ |
|---|---|---|
| `display_text` | العرض، المراجعة، وتحليل الخطأ | لا؛ تبقى كما وصلت |
| `model_text` | التدريب/البحث وفق profile معلنة | مشتقة ويمكن إعادة بنائها |

**Conservative** يحافظ على التشكيل وأشكال الألف، مع Unicode cleanup والتطويل وPII masking والمسافات.  
**Search** يضيف إزالة التشكيل وتوحيد أشكال الألف والياء المقصورة، لكنه لا يحول التاء المربوطة إلى هاء.

لا توجد profile صحيحة لكل المهام. تغييرها بعد بناء الفهرس يعني أن corpus وquery لم يعودا يتبعان العقد نفسه.


In [5]:
EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_RE = re.compile(r"(?<!\d)(?:\+?966|0)?5\d{8}(?!\d)")

def mask_pii(text):
    """Mask only the PII patterns defined in this course contract."""
    text = EMAIL_RE.sub("[EMAIL]", text)
    return PHONE_RE.sub("[PHONE]", text)


def normalise_arabic_record(text, profile="search"):
    """Return display/model copies using CAMeL Tools 1.6.0."""
    if not isinstance(text, str):
        raise TypeError("text must be str")
    if profile not in {"conservative", "search"}:
        raise ValueError("profile must be conservative or search")

    display_text = text
    model_text = mask_pii(text)
    model_text = normalize_unicode(model_text, compatibility=False)
    model_text = model_text.replace("ـ", "")
    if profile == "search":
        model_text = dediac_ar(model_text)
        model_text = normalize_alef_ar(model_text)
        model_text = normalize_alef_maksura_ar(model_text)
    model_text = " ".join(model_text.split())
    return {
        "display_text": display_text,
        "model_text": model_text,
        "profile": profile,
        "backend": "camel-tools==1.6.0",
        "profile_version": PROFILE_VERSION,
    }


In [6]:
examples = [
    normalise_arabic_record("إِدَارَةُ الحِساب", "conservative"),
    normalise_arabic_record("إِدَارَةُ الحِساب", "search"),
    normalise_arabic_record("الخـدمة   متأخرة", "search"),
]
for item in examples:
    print(item)


{'display_text': 'إِدَارَةُ الحِساب', 'model_text': 'إِدَارَةُ الحِساب', 'profile': 'conservative', 'backend': 'camel-tools==1.6.0', 'profile_version': '1.0.0'}
{'display_text': 'إِدَارَةُ الحِساب', 'model_text': 'ادارة الحساب', 'profile': 'search', 'backend': 'camel-tools==1.6.0', 'profile_version': '1.0.0'}
{'display_text': 'الخـدمة   متأخرة', 'model_text': 'الخدمة متاخرة', 'profile': 'search', 'backend': 'camel-tools==1.6.0', 'profile_version': '1.0.0'}


### توقف وفسّر | Stop and explain

1. لماذا اختلف السطران الأول والثاني؟
2. ما المعلومة التي قد نفقدها عند إزالة التشكيل؟
3. لماذا بقيت `ة` كما هي؟
4. أين يجب تخزين اسم profile وإصدارها؟


## 3) Golden tests قبل معالجة corpus

الاختبار الذهبي مثال صغير له خرج متوقع يدويًا. إذا تغيّر CAMeL Tools أو الكود، يكشف الاختبار تغير العقد قبل أن يتغير الفهرس بصمت.


In [7]:
GOLDEN_CASES = [
    {"text": "إِدَارَةُ الحِساب", "profile": "search", "expected": "ادارة الحساب"},
    {"text": "على  الطـريق", "profile": "search", "expected": "علي الطريق"},
    {"text": "إدارةُ الحساب", "profile": "conservative", "expected": "إدارةُ الحساب"},
    {"text": "راسل test@example.com", "profile": "search", "expected": "راسل [EMAIL]"},
]

golden_results = []
for case in GOLDEN_CASES:
    actual = normalise_arabic_record(case["text"], case["profile"])["model_text"]
    passed = actual == case["expected"]
    golden_results.append({**case, "actual": actual, "passed": passed})
    print("PASS" if passed else "FAIL", case["text"], "→", actual)

assert all(item["passed"] for item in golden_results)
print("GOLDEN_TESTS=PASS")


PASS إِدَارَةُ الحِساب → ادارة الحساب
PASS على  الطـريق → علي الطريق
PASS إدارةُ الحساب → إدارةُ الحساب
PASS راسل test@example.com → راسل [EMAIL]
GOLDEN_TESTS=PASS


## 4) تطبيق profile وتدقيق التنوع | Apply and audit

نطبق `search` على النصوص العربية المكتوبة بالحرف العربي. Arabizi يبقى في lane مستقل لأن تحويله يحتاج transliteration/evaluation موثقين؛ لا نحوله تلقائيًا بقواعد تخمينية.


In [8]:
processed_rows = []
for row in rows:
    if row["variant"] == "Arabizi":
        record = {
            "display_text": row["text"],
            "model_text": row["text"].strip(),
            "profile": "arabizi_passthrough",
            "backend": "none",
            "profile_version": PROFILE_VERSION,
        }
    else:
        record = normalise_arabic_record(row["text"], profile="search")
    processed_rows.append({**row, **record})

counts = Counter(row["variant"] for row in processed_rows)
print("COURSE_FIXTURE variant counts:", dict(counts))
for row in processed_rows[:5]:
    print(row["record_id"], row["display_text"], "→", row["model_text"])

assert all(row["display_text"] == original["text"] for row, original in zip(processed_rows, rows))


COURSE_FIXTURE variant counts: {'Gulf': 9, 'MSA': 9, 'Arabizi': 2}
A-001 وش سبب تعليق البوابة كل ما أرفع الملف؟ → وش سبب تعليق البوابة كل ما ارفع الملف؟
A-002 ما وصلني رمز التحقق للحين → ما وصلني رمز التحقق للحين
A-003 الباص تأخر علينا والموعد راح → الباص تاخر علينا والموعد راح
A-004 وين ألقى مسار الحافلة الجديد؟ → وين القي مسار الحافلة الجديد؟
A-005 أبي أغير موعد العيادة لبكرة → ابي اغير موعد العيادة لبكرة


In [9]:
def arabizi_candidate(text):
    # Transparent routing heuristic; this is not dialect identification.
    latin = sum(character.isascii() and character.isalpha() for character in text)
    arabizi_digits = sum(character in "2356789" for character in text)
    arabic = sum("\u0600" <= character <= "\u06ff" for character in text)
    return latin >= 3 and arabizi_digits >= 1 and arabic == 0

routed = [(row["record_id"], arabizi_candidate(row["text"])) for row in rows]
print("Arabizi heuristic candidates:", [item for item in routed if item[1]])
assert {record_id for record_id, flag in routed if flag} == {"A-019", "A-020"}


Arabizi heuristic candidates: [('A-019', True), ('A-020', True)]


## 5) قرار النموذج العربي | Arabic model decision

املأ هذه البطاقة في `DECISIONS.md` قبل الانتقال إلى البحث:

- **Task:** classification / NER / QA / retrieval
- **Checkpoint/model card:** الاسم والرابط والإصدار
- **Coverage:** MSA؟ لهجة؟ متعدد اللغات؟
- **Tokenizer evidence:** fertility وtruncation على عينتك
- **Preprocessing contract:** profile + version
- **License and limits:** من model card، لا من الذاكرة

🔵 **Explore:** قارن tokenizer لـCAMeLBERT/AraBERT أو شغّل CAMeL DialectIdentifier بعد تنزيل بياناته الرسمية. هذه الإضافة لا تستبدل Core ولا تجعل وسم `variant` تنبؤًا.


In [10]:
# حفظ دليل صغير قابل للمراجعة؛ لا يحتوي النصوص الخام الكاملة.
artifact = {
    "artifact": "bayan_arabic_profile",
    "data_kind": DATA_KIND,
    "profile": "search",
    "profile_version": PROFILE_VERSION,
    "backend": "camel-tools==1.6.0",
    "rules": [
        "preserve_display_copy",
        "mask_course_pii_patterns",
        "unicode_normalize_no_compatibility",
        "remove_tatweel_and_diacritics",
        "fold_alef_and_alef_maksura",
        "preserve_teh_marbuta",
    ],
    "golden_cases_passed": sum(item["passed"] for item in golden_results),
    "fixture_rows": len(processed_rows),
    "variant_counts_are_predictions": False,
}
Path("bayan_arabic_profile.json").write_text(
    json.dumps(artifact, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(artifact, ensure_ascii=False, indent=2))


{
  "artifact": "bayan_arabic_profile",
  "data_kind": "COURSE_FIXTURE",
  "profile": "search",
  "profile_version": "1.0.0",
  "backend": "camel-tools==1.6.0",
  "rules": [
    "preserve_display_copy",
    "mask_course_pii_patterns",
    "unicode_normalize_no_compatibility",
    "remove_tatweel_and_diacritics",
    "fold_alef_and_alef_maksura",
    "preserve_teh_marbuta"
  ],
  "golden_cases_passed": 4,
  "fixture_rows": 20,
  "variant_counts_are_predictions": false
}


## 6) مقارنة لهجية مصغرة / Dialect-aware fine-tuning smoke

هذه المقارنة تنفذ تدريبًا حقيقيًا: تجمّد معظم encoder وتدرّب آخر block ورأس التصنيف في نموذج متعدد اللغات و`CAMeLBERT-DA`. نختار epoch على أربع حالات MSA validation، ثم نفتح أربع حالات Gulf test مجمدة مرة واحدة. **أربع حالات لا تثبت تفوقًا عامًا**؛ النتيجة `MEASURED_SMOKE` لتدقيق المنهج والمسار البرمجي فقط.

In [11]:
import gc
import random
import time

import numpy as np
import torch
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
MODEL_CANDIDATES = [
    "distilbert/distilbert-base-multilingual-cased",
    "CAMeL-Lab/bert-base-arabic-camelbert-da",
]
SPLIT_IDS = {
    "train": ["A-001", "A-009", "A-003", "A-011", "A-005", "A-013", "A-006", "A-015"],
    "validation": ["A-010", "A-012", "A-014", "A-016"],
    "frozen_gulf_test": ["A-002", "A-004", "A-008", "A-007"],
}
rows_by_id = {row["record_id"]: row for row in processed_rows}
split_rows = {name: [rows_by_id[item] for item in identifiers] for name, identifiers in SPLIT_IDS.items()}
assert not (set(SPLIT_IDS["train"]) & set(SPLIT_IDS["validation"]) | set(SPLIT_IDS["train"]) & set(SPLIT_IDS["frozen_gulf_test"]) | set(SPLIT_IDS["validation"]) & set(SPLIT_IDS["frozen_gulf_test"]))
assert {row["variant"] for row in split_rows["frozen_gulf_test"]} == {"Gulf"}

LABELS = sorted({row["topic"] for row in split_rows["train"]})
label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for label, index in label2id.items()}

def train_small_comparison(model_id: str) -> dict:
    seed = 42
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(min(2, torch.get_num_threads()))
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=len(LABELS), label2id=label2id, id2label=id2label
    )
    for parameter in model.base_model.parameters():
        parameter.requires_grad = False
    layers = model.base_model.encoder.layer if hasattr(model.base_model, "encoder") else model.base_model.transformer.layer
    for parameter in layers[-1].parameters():
        parameter.requires_grad = True
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=2e-4)

    def encode(examples, include_labels=True):
        batch = tokenizer(
            [row["model_text"] for row in examples], padding=True, truncation=True,
            max_length=64, return_tensors="pt"
        )
        if include_labels:
            batch["labels"] = torch.tensor([label2id[row["topic"]] for row in examples])
        return batch

    def predict(examples):
        model.eval()
        with torch.no_grad():
            predictions = model(**encode(examples, include_labels=False)).logits.argmax(-1).tolist()
        return [id2label[index] for index in predictions]

    started = time.perf_counter()
    best_validation_f1 = -1.0
    best_epoch = 0
    best_state = None
    steps = 0
    for epoch in range(20):
        model.train()
        order = list(range(len(split_rows["train"])))
        random.Random(seed + epoch).shuffle(order)
        for start in range(0, len(order), 4):
            batch = [split_rows["train"][index] for index in order[start:start + 4]]
            optimizer.zero_grad(set_to_none=True)
            loss = model(**encode(batch)).loss
            assert torch.isfinite(loss)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            steps += 1
        validation_predictions = predict(split_rows["validation"])
        validation_f1 = f1_score(
            [row["topic"] for row in split_rows["validation"]], validation_predictions,
            labels=LABELS, average="macro", zero_division=0,
        )
        if validation_f1 > best_validation_f1:
            best_validation_f1 = float(validation_f1)
            best_epoch = epoch + 1
            best_state = {
                name: parameter.detach().cpu().clone()
                for name, parameter in model.named_parameters() if parameter.requires_grad
            }
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if name in best_state:
                parameter.copy_(best_state[name].to(parameter.device))
    test_predictions = predict(split_rows["frozen_gulf_test"])
    test_truth = [row["topic"] for row in split_rows["frozen_gulf_test"]]
    result = {
        "model_id": model_id,
        "seed": seed,
        "training_mode": "last_encoder_block_plus_head",
        "optimizer_steps": steps,
        "selected_epoch": best_epoch,
        "validation_macro_f1": best_validation_f1,
        "gulf_test_macro_f1": float(f1_score(test_truth, test_predictions, labels=LABELS, average="macro", zero_division=0)),
        "gulf_test_predictions": test_predictions,
        "elapsed_seconds": round(time.perf_counter() - started, 2),
    }
    del model, tokenizer, optimizer, best_state
    gc.collect()
    return result

comparison_results = [train_small_comparison(model_id) for model_id in MODEL_CANDIDATES]
arabic_model_comparison = {
    "result_type": "MEASURED_SMOKE",
    "split_contract": SPLIT_IDS,
    "validation_n": len(split_rows["validation"]),
    "frozen_gulf_test_n": len(split_rows["frozen_gulf_test"]),
    "one_seed_only": True,
    "results": comparison_results,
    "limitations": [
        "tiny synthetic dataset",
        "four-example validation and Gulf test slices",
        "one seed; descriptive only; no production or population claim",
    ],
}
reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)
(reports_dir / "arabic_model_comparison.json").write_text(
    json.dumps(arabic_model_comparison, ensure_ascii=False, indent=2), encoding="utf-8"
)
for result in comparison_results:
    print({
        "model_id": result["model_id"],
        "optimizer_steps": result["optimizer_steps"],
        "validation_n": len(split_rows["validation"]),
        "frozen_gulf_test_n": len(split_rows["frozen_gulf_test"]),
        "gulf_test_macro_f1": round(result["gulf_test_macro_f1"], 4),
        "result_type": "MEASURED_SMOKE",
    })
print("ARABIC_MODEL_COMPARISON=MEASURED_SMOKE")

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  439MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'model_id': 'distilbert/distilbert-base-multilingual-cased', 'optimizer_steps': 40, 'validation_n': 4, 'frozen_gulf_test_n': 4, 'gulf_test_macro_f1': 0.0, 'result_type': 'MEASURED_SMOKE'}
{'model_id': 'CAMeL-Lab/bert-base-arabic-camelbert-da', 'optimizer_steps': 40, 'validation_n': 4, 'frozen_gulf_test_n': 4, 'gulf_test_macro_f1': 0.6667, 'result_type': 'MEASURED_SMOKE'}
ARABIC_MODEL_COMPARISON=MEASURED_SMOKE


## بوابة Core | Core gate

لا تعدّل علامة النجاح يدويًا. يجب أن تنتجها الخلية بعد تحقق الشروط. بعد ظهورها احفظ الدفتر، وأضف profile واختباراتها إلى مستودعك.


In [12]:
core_checks = {
    "camel_tools_pinned": importlib.metadata.version("camel-tools") == "1.6.0",
    "dialect_comparison_ran": len(comparison_results) == 2 and all(item["optimizer_steps"] > 0 for item in comparison_results),
    "display_copy_preserved": all(
        item["display_text"] == row["text"]
        for item, row in zip(processed_rows, rows)
    ),
    "golden_tests": all(item["passed"] for item in golden_results),
    "named_profile": artifact["profile"] == "search",
    "arabizi_not_called_classifier": artifact["variant_counts_are_predictions"] is False,
    "artifact_written": Path("bayan_arabic_profile.json").exists(),
}
assert all(core_checks.values()), core_checks
print(core_checks)
print("DAY3_NOTEBOOK5_CORE=PASS")


{'camel_tools_pinned': True, 'dialect_comparison_ran': True, 'display_copy_preserved': True, 'golden_tests': True, 'named_profile': True, 'arabizi_not_called_classifier': True, 'artifact_written': True}
DAY3_NOTEBOOK5_CORE=PASS


## بعد المختبر | After the lab

1. انسخ الكود الناضج إلى `src/bayan/arabic_profiles.py`.
2. أضف golden tests إلى `tests/`.
3. وثق القرار في `DECISIONS.md`.
4. استخدم commit: `feat: add Arabic NLP profiles and tests`.
5. انتقل إلى [Notebook 06 — Semantic Search](06_semantic_search.ipynb).
